
# 05 — Compare Baselines

Objectif :

Comparer proprement plusieurs agents sur les scénarios publics du challenge :

- agents fournis par le dépôt ;
- ton agent `my_agent.py` s'il existe ;
- éventuellement les agents générés par le notebook de tuning.

Le notebook utilise les scripts officiels :

```bash
src/test_agent_validity.py
src/evaluate_submission.py
```

La logique est robuste :
- détection automatique de la racine du dépôt ;
- vérification de validité avant évaluation ;
- parsing automatique des sorties ;
- conservation des sorties brutes pour debug ;
- comparaison par score, success rate et average steps.


In [1]:

from pathlib import Path
import subprocess
import re
import json
import time
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



# 1. Détection de la racine du dépôt


In [2]:

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "evaluate_submission.py").exists() and (p / "src" / "test_agent_validity.py").exists():
            return p
    raise FileNotFoundError(
        "Impossible de trouver la racine du dépôt. "
        "Lance ce notebook depuis le dossier du repo ou depuis notebooks/."
    )

REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
AGENTS_DIR = SRC_DIR / "agents"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("REPO_ROOT  =", REPO_ROOT)
print("SRC_DIR    =", SRC_DIR)
print("AGENTS_DIR =", AGENTS_DIR)
print("RESULTS_DIR=", RESULTS_DIR)


REPO_ROOT  = /home/onyxia/work/stable_v2_RL_sailing_challenge
SRC_DIR    = /home/onyxia/work/stable_v2_RL_sailing_challenge/src
AGENTS_DIR = /home/onyxia/work/stable_v2_RL_sailing_challenge/src/agents
RESULTS_DIR= /home/onyxia/work/stable_v2_RL_sailing_challenge/results



# 2. Détection des agents disponibles

Le dépôt contient normalement :

- `agent_super_naive.py`
- `agent_trained_example.py`
- `my_agent.py` si tu l'as généré
- éventuellement `agent_tune_*.py` ou `agent_final_*.py`


In [3]:

def list_agent_files(include_tuning=False):
    agent_files = []

    preferred = [
        "agent_super_naive.py",
        "agent_trained_example.py",
        "my_agent.py",
    ]

    for name in preferred:
        p = AGENTS_DIR / name
        if p.exists():
            agent_files.append(p)

    if include_tuning:
        patterns = [
            "agent_tune_*.py",
            "agent_final_*.py",
        ]
        for pattern in patterns:
            for p in sorted(AGENTS_DIR.glob(pattern)):
                if p not in agent_files:
                    agent_files.append(p)

    return agent_files

agent_files = list_agent_files(include_tuning=True)

print("Agents found:")
for p in agent_files:
    print("-", p.relative_to(REPO_ROOT))


Agents found:
- src/agents/agent_super_naive.py
- src/agents/agent_trained_example.py
- src/agents/my_agent.py
- src/agents/agent_tune_000.py
- src/agents/agent_tune_001.py
- src/agents/agent_tune_002.py
- src/agents/agent_tune_003.py
- src/agents/agent_tune_004.py
- src/agents/agent_tune_005.py
- src/agents/agent_tune_006.py
- src/agents/agent_tune_007.py
- src/agents/agent_tune_008.py
- src/agents/agent_tune_009.py
- src/agents/agent_tune_010.py
- src/agents/agent_tune_011.py
- src/agents/agent_final_002.py
- src/agents/agent_final_006.py
- src/agents/agent_final_010.py



# 3. Fonctions robustes pour exécuter les scripts officiels


In [4]:

def run_command(cmd, cwd=SRC_DIR, timeout=900):
    start = time.time()
    completed = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - start

    return {
        "returncode": completed.returncode,
        "stdout": completed.stdout,
        "stderr": completed.stderr,
        "elapsed": elapsed,
        "cmd": " ".join(str(x) for x in cmd),
    }

def validate_agent(agent_path, verbose=False):
    cmd = ["python", "test_agent_validity.py", str(agent_path)]
    if verbose:
        cmd.append("--verbose")

    res = run_command(cmd, cwd=SRC_DIR, timeout=180)

    ok_markers = [
        "SUCCESS",
        "success",
        "valid",
        "passed",
        "✅",
    ]

    text = (res["stdout"] + "\n" + res["stderr"])
    valid = (res["returncode"] == 0) and any(m in text for m in ok_markers)

    return valid, res



# 4. Valider tous les agents


In [5]:

validity_rows = []

for agent_path in agent_files:
    valid, res = validate_agent(agent_path, verbose=False)

    validity_rows.append({
        "agent": agent_path.name,
        "path": str(agent_path),
        "valid": valid,
        "returncode": res["returncode"],
        "elapsed_sec": res["elapsed"],
        "stdout": res["stdout"],
        "stderr": res["stderr"],
    })

validity_df = pd.DataFrame(validity_rows)

display(validity_df[["agent", "valid", "returncode", "elapsed_sec"]])

for _, row in validity_df.iterrows():
    if not row["valid"]:
        print("\n" + "=" * 80)
        print("INVALID OR UNCERTAIN:", row["agent"])
        print("STDOUT:\n", row["stdout"])
        print("STDERR:\n", row["stderr"])


,agent,valid,returncode,elapsed_sec
0,agent_super_naive.py,True,0,1.569212
1,agent_trained_example.py,True,0,1.561823
2,my_agent.py,True,0,1.859411
3,agent_tune_000.py,True,0,1.847868
4,agent_tune_001.py,True,0,1.835581
5,agent_tune_002.py,True,0,1.997453
6,agent_tune_003.py,True,0,1.927653
7,agent_tune_004.py,True,0,2.050980
8,agent_tune_005.py,True,0,1.689399
9,agent_tune_006.py,True,0,1.793862



# 5. Parser les sorties de `evaluate_submission.py`

Le script officiel imprime normalement une table du type :

```text
training_1   | Success: 100.00% | Reward: 78.50 ± 3.10 | Steps: 48.0 ± 5.2
```

Le parser ci-dessous accepte quelques variantes d'espacement.


In [6]:

RESULT_RE = re.compile(
    r"(?P<scenario>training_\\d+|test)\\s*(?:\\(TEST\\))?\\s*\\|\\s*"
    r"Success:\\s*(?P<success>[0-9.]+)%\\s*\\|\\s*"
    r"Reward:\\s*(?P<reward>-?[0-9.]+)\\s*±\\s*(?P<std_reward>[0-9.]+)\\s*\\|\\s*"
    r"Steps:\\s*(?P<steps>[0-9.]+)\\s*±\\s*(?P<std_steps>[0-9.]+)"
)

def parse_eval_output(text):
    rows = []
    for line in text.splitlines():
        m = RESULT_RE.search(line)
        if m:
            d = m.groupdict()
            rows.append({
                "scenario": d["scenario"],
                "success_rate": float(d["success"]) / 100.0,
                "mean_reward": float(d["reward"]),
                "std_reward": float(d["std_reward"]),
                "mean_steps": float(d["steps"]),
                "std_steps": float(d["std_steps"]),
            })
    return rows

def evaluate_agent(agent_path, num_seeds=20, start_seed=1, scenario=None, verbose=False, timeout=1200):
    cmd = [
        "python", "evaluate_submission.py", str(agent_path),
        "--seeds", str(start_seed),
        "--num-seeds", str(num_seeds),
    ]

    if scenario is not None:
        cmd += ["--wind_scenario", scenario]

    if verbose:
        cmd += ["--verbose"]

    res = run_command(cmd, cwd=SRC_DIR, timeout=timeout)
    rows = parse_eval_output(res["stdout"])

    return rows, res



# 6. Évaluation rapide des agents valides

Commence avec peu de seeds pour vérifier que tout fonctionne.

Tu peux ensuite augmenter `NUM_SEEDS_FAST`.


In [7]:

NUM_SEEDS_FAST = 10
START_SEED = 1

valid_agents = [
    Path(row["path"])
    for _, row in validity_df.iterrows()
    if row["valid"]
]

all_eval_rows = []
raw_outputs = {}

for agent_path in valid_agents:
    print(f"Evaluating {agent_path.name} ...")

    rows, res = evaluate_agent(
        agent_path,
        num_seeds=NUM_SEEDS_FAST,
        start_seed=START_SEED,
        scenario=None,
        verbose=False,
        timeout=1200,
    )

    raw_outputs[agent_path.name] = {
        "cmd": res["cmd"],
        "returncode": res["returncode"],
        "stdout": res["stdout"],
        "stderr": res["stderr"],
        "elapsed": res["elapsed"],
    }

    if res["returncode"] != 0 or len(rows) == 0:
        print("  Evaluation failed or parsing failed.")
        print("  Return code:", res["returncode"])
        print("  STDOUT:\n", res["stdout"])
        print("  STDERR:\n", res["stderr"])
        continue

    for r in rows:
        r["agent"] = agent_path.name
        r["agent_path"] = str(agent_path)
        r["num_seeds"] = NUM_SEEDS_FAST
        r["elapsed_sec"] = res["elapsed"]
        all_eval_rows.append(r)

fast_results = pd.DataFrame(all_eval_rows)

if len(fast_results) > 0:
    display(fast_results.sort_values(["scenario", "mean_reward"], ascending=[True, False]))
else:
    print("No parsed evaluation results. Inspect raw_outputs.")


Evaluating agent_super_naive.py ...
  Evaluation failed or parsing failed.
  Return code: 0
  STDOUT:
 Loaded agent: SuperNaiveAgent

Evaluating on 3 wind scenarios with 10 seeds
Agent: SuperNaiveAgent
Maximum steps per episode: 500

WIND_SCENARIO    | SUCCESS RATE | MEAN REWARD       | MEAN STEPS
---------------------------------------------------------------------------
training_1   | Success: 100.00% | Reward: 39.86 ± 5.66 | Steps: 186.7 ± 30.4
training_2   | Success: 100.00% | Reward: 40.44 ± 2.63 | Steps: 182.0 ± 12.1
training_3   | Success: 100.00% | Reward: 38.01 ± 0.66 | Steps: 194.0 ± 3.5
---------------------------------------------------------------------------
OVERALL      | Success: 100.00% ± 0.00%
Reward: 39.44 ± 1.04
Steps: 187.6 ± 4.9

  STDERR:
 
Evaluating agent_trained_example.py ...
  Evaluation failed or parsing failed.
  Return code: 0
  STDOUT:
 Loaded agent: QLearningTrainedAgent

Evaluating on 3 wind scenarios with 10 seeds
Agent: QLearningTrainedAgent
Maximum 

KeyboardInterrupt: 


# 7. Sauvegarder les sorties brutes

Très utile si un agent échoue ou si le parsing doit être adapté.


In [ ]:

raw_path = RESULTS_DIR / "compare_baselines_raw_outputs.json"
raw_path.write_text(json.dumps(raw_outputs, indent=2), encoding="utf-8")

if len(fast_results) > 0:
    csv_path = RESULTS_DIR / "compare_baselines_fast_results.csv"
    fast_results.to_csv(csv_path, index=False)
    print("Saved:", csv_path)

print("Saved raw outputs:", raw_path)



# 8. Agréger les résultats par agent

On calcule :

- score moyen sur les scénarios publics ;
- pire score public ;
- success rate moyen ;
- pire success rate ;
- steps moyens ;
- pire nombre de steps.


In [ ]:

def aggregate_results(df):
    if len(df) == 0:
        return pd.DataFrame()

    agg = (
        df.groupby("agent")
        .agg(
            avg_reward=("mean_reward", "mean"),
            min_reward=("mean_reward", "min"),
            avg_success=("success_rate", "mean"),
            min_success=("success_rate", "min"),
            avg_steps=("mean_steps", "mean"),
            max_steps=("mean_steps", "max"),
            total_eval_time_sec=("elapsed_sec", "sum"),
        )
        .reset_index()
    )

    agg = agg.sort_values(
        ["avg_success", "avg_reward", "avg_steps"],
        ascending=[False, False, True],
    )

    return agg

fast_summary = aggregate_results(fast_results)

display(fast_summary)



# 9. Visualisations comparatives


In [ ]:

if len(fast_results) > 0:
    pivot_reward = fast_results.pivot(index="scenario", columns="agent", values="mean_reward")
    display(pivot_reward)

    ax = pivot_reward.plot(kind="bar", figsize=(10, 5))
    ax.set_ylabel("Mean discounted reward")
    ax.set_title("Mean reward by scenario and agent")
    ax.grid(axis="y")
    plt.xticks(rotation=0)
    plt.show()

    pivot_steps = fast_results.pivot(index="scenario", columns="agent", values="mean_steps")
    ax = pivot_steps.plot(kind="bar", figsize=(10, 5))
    ax.set_ylabel("Mean steps")
    ax.set_title("Average steps by scenario and agent")
    ax.grid(axis="y")
    plt.xticks(rotation=0)
    plt.show()

    pivot_success = fast_results.pivot(index="scenario", columns="agent", values="success_rate")
    ax = pivot_success.plot(kind="bar", figsize=(10, 5))
    ax.set_ylabel("Success rate")
    ax.set_title("Success rate by scenario and agent")
    ax.grid(axis="y")
    plt.xticks(rotation=0)
    plt.show()
else:
    print("No results to plot.")



# 10. Évaluation robuste des meilleurs agents

On sélectionne les meilleurs agents de l'évaluation rapide, puis on les réévalue avec davantage de seeds.

Le leaderboard caché utilise 50 seeds.  
Ici, mets `NUM_SEEDS_FINAL = 50` pour une validation sérieuse.


In [ ]:

TOP_K = 3
NUM_SEEDS_FINAL = 50

top_agents = []
if len(fast_summary) > 0:
    top_agents = fast_summary.head(TOP_K)["agent"].tolist()

print("Top agents selected:", top_agents)

final_rows = []
final_raw_outputs = {}

for agent_name in top_agents:
    agent_path = AGENTS_DIR / agent_name
    print(f"Final evaluation for {agent_name} ...")

    rows, res = evaluate_agent(
        agent_path,
        num_seeds=NUM_SEEDS_FINAL,
        start_seed=1,
        scenario=None,
        verbose=False,
        timeout=1800,
    )

    final_raw_outputs[agent_name] = {
        "cmd": res["cmd"],
        "returncode": res["returncode"],
        "stdout": res["stdout"],
        "stderr": res["stderr"],
        "elapsed": res["elapsed"],
    }

    if res["returncode"] != 0 or len(rows) == 0:
        print("  Failed or parsing failed.")
        print("STDOUT:", res["stdout"])
        print("STDERR:", res["stderr"])
        continue

    for r in rows:
        r["agent"] = agent_name
        r["agent_path"] = str(agent_path)
        r["num_seeds"] = NUM_SEEDS_FINAL
        r["elapsed_sec"] = res["elapsed"]
        final_rows.append(r)

final_results = pd.DataFrame(final_rows)

if len(final_results) > 0:
    display(final_results.sort_values(["scenario", "mean_reward"], ascending=[True, False]))
    final_summary = aggregate_results(final_results)
    display(final_summary)
else:
    final_summary = pd.DataFrame()
    print("No final results.")



# 11. Sauvegarder les résultats finaux


In [ ]:

if len(final_results) > 0:
    final_csv = RESULTS_DIR / "compare_baselines_final_results.csv"
    final_summary_csv = RESULTS_DIR / "compare_baselines_final_summary.csv"

    final_results.to_csv(final_csv, index=False)
    final_summary.to_csv(final_summary_csv, index=False)

    print("Saved:", final_csv)
    print("Saved:", final_summary_csv)

final_raw_path = RESULTS_DIR / "compare_baselines_final_raw_outputs.json"
final_raw_path.write_text(json.dumps(final_raw_outputs, indent=2), encoding="utf-8")
print("Saved:", final_raw_path)



# 12. Diagnostiquer les scénarios faibles

Un agent robuste ne doit pas seulement avoir un bon score moyen.  
Il doit aussi éviter de s'effondrer sur un scénario public.

On regarde donc :
- le scénario avec le score minimal ;
- le scénario avec le success rate minimal ;
- le scénario avec le plus grand nombre de steps.


In [ ]:

diagnostic_df = final_results if len(final_results) > 0 else fast_results

if len(diagnostic_df) > 0:
    for agent, sub in diagnostic_df.groupby("agent"):
        worst_reward = sub.loc[sub["mean_reward"].idxmin()]
        worst_success = sub.loc[sub["success_rate"].idxmin()]
        worst_steps = sub.loc[sub["mean_steps"].idxmax()]

        print("\n" + "=" * 80)
        print("Agent:", agent)
        print("Worst reward scenario :", worst_reward["scenario"], "reward=", worst_reward["mean_reward"])
        print("Worst success scenario:", worst_success["scenario"], "success=", worst_success["success_rate"])
        print("Worst steps scenario  :", worst_steps["scenario"], "steps=", worst_steps["mean_steps"])
else:
    print("No diagnostics available.")



# 13. Comparaison avec le leaderboard

Le premier annoncé a environ :

- score : 79.47 ;
- average steps : 45.86 ;
- success rate : 1.0.

Comme le score est :
\[
100 \times 0.995^{steps},
\]
un agent à 46 steps avec 100% de succès obtient environ 79.4.

Cette cellule calcule l'équivalent théorique.


In [ ]:

def theoretical_score_from_steps(steps, success_rate=1.0):
    return 100.0 * success_rate * (0.995 ** steps)

for steps in [40, 45.86, 50, 60, 70, 100]:
    print(f"steps={steps:6.2f} -> score≈{theoretical_score_from_steps(steps):.2f}")

if len(final_summary) > 0:
    tmp = final_summary.copy()
elif len(fast_summary) > 0:
    tmp = fast_summary.copy()
else:
    tmp = pd.DataFrame()

if len(tmp) > 0:
    tmp["theoretical_score_from_avg_steps"] = tmp.apply(
        lambda r: theoretical_score_from_steps(r["avg_steps"], r["avg_success"]),
        axis=1,
    )
    display(tmp[[
        "agent",
        "avg_reward",
        "avg_success",
        "avg_steps",
        "theoretical_score_from_avg_steps",
    ]])



# 14. Sélection finale

À ce stade :

- si `my_agent.py` domine les baselines sur les 3 scénarios publics ;
- si son success rate est proche de 1 ;
- si ses steps moyens sont faibles ;
- et si son temps d'évaluation reste raisonnable ;

alors on peut passer au notebook suivant :

```text
06_visualize_failures.ipynb
```

Sinon, il faut retourner au notebook :

```text
04_parameter_tuning.ipynb
```

pour améliorer :
- les waypoints ;
- les pénalités ;
- l'horizon ;
- la stratégie gauche/droite.
